# Centralized baseline

training the "normal" way first, all data in one place, before trying federated.
this is the number to beat later.

trying isolation forest and a small autoencoder, both trained only on benign traffic,
tested on a mix of benign + attack

In [1]:
# loading what notebook 1 saved

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv("/content/drive/MyDrive/nbaiot-project/processed/combined_labeled.csv")
df.shape, df["label"].value_counts()

Mounted at /content/drive


((1344470, 118),
 label
 1    1059984
 0     284486
 Name: count, dtype: int64)

training on benign only, testing on a mix of benign + attack. this is the realistic
setup since you don't actually have labeled attacks ahead of time in practice

In [2]:
feature_cols = [c for c in df.columns if c not in ["label", "attack_type", "device"]]
X = df[feature_cols].values
y = df["label"].values

benign_mask = y == 0
X_benign = X[benign_mask]
X_train, X_benign_holdout = train_test_split(X_benign, test_size=0.3, random_state=42)
X_attack = X[~benign_mask]

X_test = np.vstack([X_benign_holdout, X_attack])
y_test = np.hstack([np.zeros(len(X_benign_holdout)), np.ones(len(X_attack))])

scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("train (benign only):", X_train_scaled.shape)
print("test:", X_test_scaled.shape, "| attack ratio:", round(y_test.mean(), 3))

train (benign only): (199140, 115)
test: (1145330, 115) | attack ratio: 0.925


## Isolation forest

In [3]:
# quick baseline, contamination=0.1 is just a guess, can tune later

!pip install -q pyod
from pyod.models.iforest import IForest

iforest = IForest(contamination=0.1, random_state=42)
iforest.fit(X_train_scaled)

scores_if = iforest.decision_function(X_test_scaled)
preds_if = iforest.predict(X_test_scaled)

print(classification_report(y_test, preds_if, target_names=["benign", "attack"]))
print("ROC-AUC:", round(roc_auc_score(y_test, scores_if), 4))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.3/59.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 414.5/414.5 kB 29.3 MB/s eta 0:00:00
              precision    recall  f1-score   support

      benign       1.00      0.90      0.95     85346
      attack       0.99      1.00      1.00   1059984

    accuracy                           0.99   1145330
   macro avg       1.00      0.95      0.97   1145330
weighted avg       0.99      0.99      0.99   1145330

ROC-AUC: 0.9762


## autoencoder

trained to reconstruct benign traffic only. high reconstruction error at test time
= probably an anomaly

In [4]:
# small autoencoder, nothing fancy

import torch
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"

class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, latent_dim), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.ReLU(),
            nn.Linear(64, input_dim),
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

model = Autoencoder(X_train_scaled.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)

n_epochs = 30
batch_size = 256
for epoch in range(n_epochs):
    perm = torch.randperm(X_train_t.size(0))
    total_loss = 0

    # 30 epochs was enough to converge, didn't need more

    for i in range(0, X_train_t.size(0), batch_size):
        idx = perm[i:i+batch_size]
        batch = X_train_t[idx]
        optimizer.zero_grad()
        recon = model(batch)
        loss = criterion(recon, batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch+1) % 5 == 0:
        print(f"epoch {epoch+1}/{n_epochs} — loss {total_loss:.4f}")

epoch 5/30 — loss 25.0624
epoch 10/30 — loss 18.4421
epoch 15/30 — loss 15.8467
epoch 20/30 — loss 14.1420
epoch 25/30 — loss 13.0243
epoch 30/30 — loss 12.1562


In [5]:
# threshold = 95th percentile of train error, standard approach for this

model.eval()
with torch.no_grad():
    X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)
    recon = model(X_test_t)
    recon_error = torch.mean((X_test_t - recon) ** 2, dim=1).cpu().numpy()

print("ROC-AUC:", round(roc_auc_score(y_test, recon_error), 4))

with torch.no_grad():
    train_recon = model(X_train_t)
    train_error = torch.mean((X_train_t - train_recon) ** 2, dim=1).cpu().numpy()
threshold = np.percentile(train_error, 95)

preds_ae = (recon_error > threshold).astype(int)
print(classification_report(y_test, preds_ae, target_names=["benign", "attack"]))

ROC-AUC: 1.0
              precision    recall  f1-score   support

      benign       1.00      0.95      0.97     85346
      attack       1.00      1.00      1.00   1059984

    accuracy                           1.00   1145330
   macro avg       1.00      0.97      0.99   1145330
weighted avg       1.00      1.00      1.00   1145330



saving both results, need these again in the federated notebook for comparison

In [6]:
import json
import os

save_dir = "/content/drive/MyDrive/nbaiot-project/processed"
os.makedirs(save_dir, exist_ok=True)

baseline_results = {
    "isolation_forest": {"roc_auc": float(roc_auc_score(y_test, scores_if))},
    "autoencoder": {"roc_auc": float(roc_auc_score(y_test, recon_error))},
}
with open(os.path.join(save_dir, "baseline_results.json"), "w") as f:
    json.dump(baseline_results, f, indent=2)
baseline_results

{'isolation_forest': {'roc_auc': 0.9762494846705717},
 'autoencoder': {'roc_auc': 0.9999945608815071}}

autoencoder came out ahead of isolation forest here (0.9999 vs 0.976 roc-auc).
makes sense honestly, the benign vs attack traffic was already looking pretty
separable in the notebook 1 plots, so a model with more capacity can basically
nail it. good baseline to compare the federated version against next

Next up: `03_federated_flower.ipynb`, same problem, but federated across devices.